# Binary Classification Under Noise

Exploring how label noise, loss functions, and metrics affect a PyTorch logistic regression model trained on synthetic datasets.

## Experiment Roadmap

- Generate 2D binary datasets with varying levels of label noise using `make_classification`.
- Train logistic regression models with the reusable `Architecture` class.
- Compare training with `nn.BCELoss` versus `nn.BCEWithLogitsLoss`.
- Visualize decision boundaries and confusion matrices for each configuration.
- Quantify performance via accuracy, precision, recall, F1-score, and discuss stability trade-offs.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

ROOT = Path('..').resolve()
sys.path.append(str(ROOT / 'src'))

from architecture import Architecture

RESULTS_DIR = ROOT / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

plt.style.use('seaborn-v0_8')
sns.set_context('talk')

torch.manual_seed(42)
np.random.seed(42)

## Data Generation Pipeline

We synthesize 2D datasets whose separability is gradually degraded by increasing the `flip_y` noise parameter. This keeps the geometry simple enough for 2D decision boundary plots.

In [2]:
def generate_dataset(noise_level: float, n_samples: int = 600, class_sep: float = 1.5):
    """Create a 2D binary classification dataset with controllable label noise."""
    X, y = make_classification(
        n_samples=n_samples,
        n_features=2,
        n_redundant=0,
        n_informative=2,
        n_clusters_per_class=1,
        class_sep=class_sep,
        flip_y=noise_level,
        random_state=42,
    )
    return X.astype(np.float32), y.astype(np.int64)


def prepare_dataloaders(
    X: np.ndarray,
    y: np.ndarray,
    batch_size: int = 32,
    test_size: float = 0.3,
    seed: int = 42,
):
    """Standardize features, split into train/validation, and build PyTorch loaders."""
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=test_size, stratify=y, random_state=seed
    )

    scaler = StandardScaler().fit(X_train)
    X_train_scaled = scaler.transform(X_train).astype(np.float32)
    X_val_scaled = scaler.transform(X_val).astype(np.float32)

    y_train_float = y_train.astype(np.float32).reshape(-1, 1)
    y_val_float = y_val.astype(np.float32).reshape(-1, 1)

    train_dataset = TensorDataset(
        torch.tensor(X_train_scaled, dtype=torch.float32),
        torch.tensor(y_train_float, dtype=torch.float32),
    )
    val_dataset = TensorDataset(
        torch.tensor(X_val_scaled, dtype=torch.float32),
        torch.tensor(y_val_float, dtype=torch.float32),
    )

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    splits = {
        'X_train': X_train,
        'X_val': X_val,
        'y_train': y_train,
        'y_val': y_val,
        'X_train_scaled': X_train_scaled,
        'X_val_scaled': X_val_scaled,
    }

    return train_loader, val_loader, scaler, splits

## Model, Visualization, and Metric Utilities

We reuse the provided `Architecture` class for training loops, define a lightweight logistic regression model, and add helpers to map predictions back to plots and metrics.

In [3]:
class LogisticRegression(nn.Module):
    def __init__(self, input_dim: int = 2, apply_sigmoid: bool = False):
        super().__init__()
        self.linear = nn.Linear(input_dim, 1)
        self.apply_sigmoid = apply_sigmoid

    def forward(self, x):
        logits = self.linear(x)
        if self.apply_sigmoid:
            return torch.sigmoid(logits)
        return logits


def sigmoid_np(x):
    return 1.0 / (1.0 + np.exp(-x))


def plot_decision_boundary(architecture, scaler, X_full, y_full, noise_level, loss_name):
    x_min, x_max = X_full[:, 0].min() - 1.0, X_full[:, 0].max() + 1.0
    y_min, y_max = X_full[:, 1].min() - 1.0, X_full[:, 1].max() + 1.0
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300)
    )
    grid = np.c_[xx.ravel(), yy.ravel()].astype(np.float32)
    grid_scaled = scaler.transform(grid)
    preds = architecture.predict(grid_scaled)
    if loss_name == 'BCEWithLogitsLoss':
        probs = sigmoid_np(preds)
    else:
        probs = preds
    probs = probs.reshape(xx.shape)

    fig, ax = plt.subplots(figsize=(8, 6))
    contour = ax.contourf(
        xx,
        yy,
        probs,
        levels=np.linspace(0, 1, 21),
        cmap='RdBu',
        alpha=0.8,
    )
    ax.contour(xx, yy, probs, levels=[0.5], colors='k', linestyles='--', linewidths=2)
    scatter = ax.scatter(
        X_full[:, 0],
        X_full[:, 1],
        c=y_full,
        cmap=plt.cm.coolwarm,
        edgecolors='k',
        s=35,
        alpha=0.9,
    )
    ax.set_title(f'Decision Boundary | noise={noise_level:.2f} | loss={loss_name}')
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')
    ax.legend(*scatter.legend_elements(), title='Class')
    cbar = fig.colorbar(contour, ax=ax)
    cbar.set_label('P(class = 1)')

    file_path = RESULTS_DIR / f'decision_boundary_noise_{noise_level:.2f}_{loss_name}.png'
    fig.savefig(file_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    return file_path


def plot_confusion_matrix(y_true, y_pred, noise_level, loss_name):
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        cbar=False,
        ax=ax,
    )
    ax.set_title(f'Confusion Matrix | noise={noise_level:.2f} | loss={loss_name}')
    ax.set_xlabel('Predicted label')
    ax.set_ylabel('True label')
    ax.set_xticklabels(['Class 0', 'Class 1'])
    ax.set_yticklabels(['Class 0', 'Class 1'])

    file_path = RESULTS_DIR / f'confusion_noise_{noise_level:.2f}_{loss_name}.png'
    fig.savefig(file_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    return file_path


def plot_loss_curve(architecture, noise_level, loss_name):
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(architecture.losses, label='Train loss')
    if architecture.val_losses and architecture.val_losses[0] is not None:
        ax.plot(architecture.val_losses, label='Validation loss')
    ax.set_yscale('log')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title(f'Loss Curve | noise={noise_level:.2f} | loss={loss_name}')
    ax.legend()

    file_path = RESULTS_DIR / f'loss_curve_noise_{noise_level:.2f}_{loss_name}.png'
    fig.savefig(file_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    return file_path

## Training Loop Wrapper

This helper orchestrates the full experiment for a given noise level and loss function, returning metrics plus artifact paths for downstream reporting.

In [4]:
def train_and_evaluate(noise_level, loss_name, n_epochs=300, lr=0.1, batch_size=32):
    X, y = generate_dataset(noise_level=noise_level)
    train_loader, val_loader, scaler, splits = prepare_dataloaders(
        X, y, batch_size=batch_size
    )

    model = LogisticRegression(
        input_dim=X.shape[1],
        apply_sigmoid=(loss_name == 'BCELoss'),
    )
    loss_fn = nn.BCELoss() if loss_name == 'BCELoss' else nn.BCEWithLogitsLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)

    architecture = Architecture(model, loss_fn, optimizer)
    architecture.set_loaders(train_loader, val_loader)
    architecture.train(n_epochs=n_epochs, seed=42)

    X_val_scaled = splits['X_val_scaled']
    raw_preds = architecture.predict(X_val_scaled)
    if loss_name == 'BCEWithLogitsLoss':
        probs = sigmoid_np(raw_preds)
    else:
        probs = raw_preds
    probs = probs.reshape(-1)
    y_pred = (probs >= 0.5).astype(int)
    y_true = splits['y_val']

    metrics = {
        'noise_level': noise_level,
        'loss_fn': loss_name,
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1_score': f1_score(y_true, y_pred, zero_division=0),
        'final_train_loss': architecture.losses[-1],
        'final_val_loss': architecture.val_losses[-1],
        'epochs': architecture.total_epochs,
    }

    X_full = np.vstack((splits['X_train'], splits['X_val']))
    y_full = np.concatenate((splits['y_train'], splits['y_val']))

    metrics['decision_boundary_path'] = str(
        plot_decision_boundary(
            architecture, scaler, X_full, y_full, noise_level, loss_name
        ).relative_to(ROOT)
    )
    metrics['confusion_matrix_path'] = str(
        plot_confusion_matrix(y_true, y_pred, noise_level, loss_name).relative_to(ROOT)
    )
    metrics['loss_curve_path'] = str(
        plot_loss_curve(architecture, noise_level, loss_name).relative_to(ROOT)
    )

    return metrics, architecture

## Run Noise Experiments

Execute the pipeline for multiple noise settings and both loss functions. The resulting metrics table and saved figures will feed into the README and presentation.

In [5]:
noise_levels = [0.0, 0.1, 0.2, 0.3, 0.4]
loss_options = ['BCELoss', 'BCEWithLogitsLoss']
metrics_records = []

for noise in noise_levels:
    for loss_name in loss_options:
        print(f'Running experiment: noise={noise:.2f}, loss={loss_name}')
        metrics, _ = train_and_evaluate(noise, loss_name)
        metrics_records.append(metrics)

metrics_df = pd.DataFrame(metrics_records)
metrics_df.sort_values(['noise_level', 'loss_fn'], inplace=True)
metrics_path = RESULTS_DIR / 'metrics_summary.csv'
metrics_df.to_csv(metrics_path, index=False)
print(f'Exported consolidated metrics to {metrics_path.relative_to(ROOT)}')

Running experiment: noise=0.00, loss=BCELoss


Running experiment: noise=0.00, loss=BCEWithLogitsLoss


Running experiment: noise=0.10, loss=BCELoss


Running experiment: noise=0.10, loss=BCEWithLogitsLoss


Running experiment: noise=0.20, loss=BCELoss


...

Running experiment: noise=0.30, loss=BCELoss


Running experiment: noise=0.30, loss=BCEWithLogitsLoss


Running experiment: noise=0.40, loss=BCELoss


Running experiment: noise=0.40, loss=BCEWithLogitsLoss


Exported consolidated metrics to results/metrics_summary.csv


## Metrics Overview

Inspect the aggregated performance metrics to trace how noise and loss choice impact the classifier.

In [6]:
metrics_df

,noise_level,loss_fn,accuracy,precision,recall,f1_score,final_train_loss,final_val_loss,epochs,decision_boundary_path,confusion_matrix_path,loss_curve_path
0,0.0,BCELoss,0.994444,0.989011,1.000000,0.994475,0.020497,0.022263,300,results/decision_boundary_noise_0.00_BCELoss.png,results/confusion_noise_0.00_BCELoss.png,results/loss_curve_noise_0.00_BCELoss.png
1,0.0,BCEWithLogitsLoss,0.994444,0.989011,1.000000,0.994475,0.020514,0.022280,300,results/decision_boundary_noise_0.00_BCEWithLo...,results/confusion_noise_0.00_BCEWithLogitsLoss...,results/loss_curve_noise_0.00_BCEWithLogitsLos...
2,0.1,BCELoss,0.950000,0.934783,0.966292,0.950276,0.315184,0.229844,300,results/decision_boundary_noise_0.10_BCELoss.png,results/confusion_noise_0.10_BCELoss.png,results/loss_curve_noise_0.10_BCELoss.png
3,0.1,BCEWithLogitsLoss,0.950000,0.934783,0.966292,0.950276,0.315184,0.229844,300,results/decision_boundary_noise_0.10_BCEWithLo...,results/confusion_noise_0.10_BCEWithLogitsLoss...,results/loss_curve_noise_0.10_BCEWithLogitsLos...
4,0.2,BCELoss,0.911111,0.892473,0.932584,0.912088,0.351943,0.313568,300,results/decision_boundary_noise_0.20_BCELoss.png,results/confusion_noise_0.20_BCELoss.png,results/loss_curve_noise_0.20_BCELoss.png
5,0.2,BCEWithLogitsLoss,0.911111,0.892473,0.932584,0.912088,0.351943,0.313568,300,results/decision_boundary_noise_0.20_BCEWithLo...,results/confusion_noise_0.20_BCEWithLogitsLoss...,results/loss_curve_noise_0.20_BCEWithLogitsLos...
6,0.3,BCELoss,0.872222,0.858696,0.887640,0.872928,0.466036,0.413189,300,results/decision_boundary_noise_0.30_BCELoss.png,results/confusion_noise_0.30_BCELoss.png,results/loss_curve_noise_0.30_BCELoss.png
7,0.3,BCEWithLogitsLoss,0.872222,0.858696,0.887640,0.872928,0.466036,0.413189,300,results/decision_boundary_noise_0.30_BCEWithLo...,results/confusion_noise_0.30_BCEWithLogitsLoss...,results/loss_curve_noise_0.30_BCEWithLogitsLos...
8,0.4,BCELoss,0.800000,0.829268,0.755556,0.790698,0.524392,0.519075,300,results/decision_boundary_noise_0.40_BCELoss.png,results/confusion_noise_0.40_BCELoss.png,results/loss_curve_noise_0.40_BCELoss.png
9,0.4,BCEWithLogitsLoss,0.800000,0.829268,0.755556,0.790698,0.524392,0.519076,300,results/decision_boundary_noise_0.40_BCEWithLo...,results/confusion_noise_0.40_BCEWithLogitsLoss...,results/loss_curve_noise_0.40_BCEWithLogitsLos...


## Notes and Next Steps

- Examine how the decision boundary plots evolve as noise increases; document key observations in the README.
- Compare numerical stability: `BCEWithLogitsLoss` should be more stable for extreme logits, while `BCELoss` may saturate.
- Incorporate highlights from confusion matrices (e.g., shifts in false positives vs. false negatives).
- Use the exported figures (`results/` folder) inside the README and presentation slides.
- Optionally extend the study with `make_circles` or `make_moons` to showcase non-linear decision boundaries.